# LLM Exposure Index – Datenerhebung und Aufbereitung
**Masterarbeit
Autor: Ayumi Nojima  
Datum: April 2026  

---
## Outputs dieses Notebooks

| Datei | Inhalt | Kapitel |
|-------|--------|---------| 
| `processed/mu_weights.csv` | LLM-Gewichte μ_i | 5.5 |
| `processed/skill_vectors_standardized.csv` | Standardisierte Fähigkeitsvektoren | 5.7 |
| `output/dataset_H1.csv` | H1: Gruppenunterschiede (ANOVA) | 5.7 |
| `output/dataset_H2.csv` | H2: Feature Importance (Random Forest) | 5.7 |
| `output/dataset_H3.csv` | H3: Beschäftigungsveränderung (OLS) | 5.7 |
| `output/dataset_validation.csv` | Konvergente Validierung (Kuprecht-2025) | 5.7 |


## 0. Imports und Konfiguration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

# ── Repo-Root robust bestimmen ─────────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if (_cwd / "..").resolve().joinpath("data").exists() else _cwd
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = Path.cwd()
print("Repo-Root:", REPO_ROOT)

# ── Pfade ──────────────────────────────────────────────────────────────────
PROCESSED = REPO_ROOT / "data" / "processed"
ANALYSIS  = PROCESSED / "analysis_prep"
OUTPUT    = REPO_ROOT / "data" / "output"
for p in [ANALYSIS, OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Parameter ──────────────────────────────────────────────────────────────
SKILL_OVERLAP_THRESHOLD = 0.70
MAIN_GROUPS             = [1, 2, 3]
BFS_BASE_YEAR           = 2022
BFS_TARGET_YEAR         = 2024

# ── Daten aus Notebook 1 laden ─────────────────────────────────────────────
onet_pivot = pd.read_csv(PROCESSED / "onet_pivot.csv")
train_mapping = pd.read_csv(PROCESSED / "train_mapping.csv")
ch_isco       = pd.read_csv(PROCESSED / "ch_isco_clean.csv")
bfs           = pd.read_csv(PROCESSED / "bfs_clean.csv")

print(f"onet_pivot:    {onet_pivot.shape[0]} Berufe × {onet_pivot.shape[1]-1} Dimensionen ✓")
print(f"train_mapping: {len(train_mapping)} Berufe ✓")
print(f"ch_isco:       {len(ch_isco)} Klassen ✓")
print(f"bfs:           {len(bfs)} Berufsgruppen ✓")
print("Konfiguration geladen ✓")

Repo-Root: /workspaces/Master_Thesis
onet_pivot:    894 Berufe × 161 Dimensionen ✓
train_mapping: 571 Berufe ✓
ch_isco:       271 Klassen ✓
bfs:           215 Berufsgruppen ✓
Konfiguration geladen ✓


---
## 5.5 Kalibrierung der LLM-Gewichte (μ_i)

μ_i-Gewichte basieren auf Eloundou et al. (2023), adjustiert für aktuelle LLM-Benchmarks  
(OpenAI et al., 2024; DeepSeek-AI et al., 2025). Rescaling auf [0.1, 0.9] verhindert  
Extremnullgewichtungen. **Schwellenwerte:**
- μ_i > 0.7 → Substitutionsanteil (E^sub_j)  
- 0.3 ≤ μ_i ≤ 0.7 → Augmentationsanteil (E^aug_j)  
- μ_i < 0.3 → nicht in Index aufgenommen

**Zirkularitätsprävention:** Gewichte werden extern kalibriert (nicht aus Validierungsdaten  
geschätzt), um eine closed-loop-Situation zu vermeiden (vgl. Kap. 4.5.2).


In [22]:
# μ_i-Gewichte (Eloundou et al., 2023, adjustiert für GPT-4o/Claude 4/DeepSeek V3)
# Präfix-Format muss mit element_name in onet_pivot übereinstimmen
mu_weights_raw = {
    # ── Skills ──────────────────────────────────────────────────────────────
    "Skills – Reading Comprehension":           0.85,
    "Skills – Writing":                         0.88,
    "Skills – Active Listening":                0.55,
    "Skills – Speaking":                        0.45,
    "Skills – Mathematics":                     0.80,
    "Skills – Science":                         0.55,
    "Skills – Critical Thinking":               0.82,
    "Skills – Active Learning":                 0.60,
    "Skills – Learning Strategies":             0.50,
    "Skills – Monitoring":                      0.45,
    "Skills – Social Perceptiveness":           0.30,
    "Skills – Coordination":                    0.35,
    "Skills – Persuasion":                      0.40,
    "Skills – Negotiation":                     0.38,
    "Skills – Instructing":                     0.42,
    "Skills – Service Orientation":             0.35,
    "Skills – Complex Problem Solving":         0.80,
    "Skills – Operations Analysis":             0.70,
    "Skills – Technology Design":               0.55,
    "Skills – Equipment Selection":             0.20,
    "Skills – Installation":                    0.10,
    "Skills – Programming":                     0.90,
    "Skills – Quality Control Analysis":        0.50,
    "Skills – Operations Monitoring":           0.35,
    "Skills – Operation and Control":           0.15,
    "Skills – Equipment Maintenance":           0.10,
    "Skills – Troubleshooting":                 0.45,
    "Skills – Repairing":                       0.10,
    "Skills – Systems Analysis":                0.72,
    "Skills – Systems Evaluation":              0.68,
    "Skills – Time Management":                 0.45,
    "Skills – Management of Financial Resources": 0.55,
    "Skills – Management of Material Resources":  0.25,
    "Skills – Management of Personnel Resources": 0.40,
    # ── Work Activities ──────────────────────────────────────────────────────
    "Work Activities – Getting Information":                   0.80,
    "Work Activities – Processing Information":                0.85,
    "Work Activities – Analyzing Data or Information":         0.82,
    "Work Activities – Making Decisions and Solving Problems": 0.75,
    "Work Activities – Thinking Creatively":                   0.70,
    "Work Activities – Updating and Using Relevant Knowledge": 0.72,
    "Work Activities – Communicating with Supervisors, Peers, or Subordinates": 0.40,
    "Work Activities – Communicating with People Outside the Organization":     0.38,
    "Work Activities – Establishing and Maintaining Interpersonal Relationships": 0.25,
    "Work Activities – Developing Objectives and Strategies":  0.65,
    "Work Activities – Organizing, Planning, and Prioritizing Work": 0.55,
    "Work Activities – Performing Administrative Activities":  0.75,
    "Work Activities – Documenting/Recording Information":     0.80,
    "Work Activities – Interpreting the Meaning of Information for Others": 0.72,
    "Work Activities – Provide Consultation and Advice to Others": 0.60,
    # ── Knowledge ────────────────────────────────────────────────────────────
    "Knowledge – English Language":              0.85,
    "Knowledge – Mathematics":                   0.80,
    "Knowledge – Computers and Electronics":     0.82,
    "Knowledge – Administration and Management": 0.60,
    "Knowledge – Economics and Accounting":      0.70,
    "Knowledge – Law and Government":            0.65,
    "Knowledge – Medicine and Dentistry":        0.45,
    "Knowledge – Education and Training":        0.55,
}

# Rescaling auf [0.1, 0.9]
mu_min = min(mu_weights_raw.values())
mu_max = max(mu_weights_raw.values())
mu_weights = {
    k: round(0.1 + (v - mu_min) / (mu_max - mu_min) * 0.8, 4)
    for k, v in mu_weights_raw.items()
}

mu_df = pd.DataFrame([
    {
        "element_name":  k,
        "mu_i":          v,
        "exposure_type": "substitution" if v > 0.7 else ("augmentation" if v >= 0.3 else "negligible")
    }
    for k, v in mu_weights.items()
])

print(f"μ_i-Gewichte definiert: {len(mu_df)} Dimensionen")
print(mu_df["exposure_type"].value_counts().to_string())

mu_df.to_csv(ANALYSIS / "mu_weights.csv", index=False)
print("\nGespeichert → data/processed/analysis_prep/mu_weights.csv ✓")


μ_i-Gewichte definiert: 57 Dimensionen
exposure_type
augmentation    32
substitution    18
negligible       7

Gespeichert → data/processed/analysis_prep/mu_weights.csv ✓


---
## 5.6 Berechnung des Exposure-Index E_j

$$E_j = \sum_{i=1}^{n} \mu_i \cdot w_{ij} \cdot s_{ij}$$

- $w_{ij}$ = Min-Max-normalisierter Importance-Wert (O\*NET)
- $\mu_i$ = LLM-Gewicht der Fähigkeitsdimension i
- $s_{ij}$ = Skill-Overlap-Koeffizient (Pearson-Korrelation zwischen  
  berufsspezifischem Fähigkeitsprofil und μ_i-Referenzprofil)  
  → Berufe mit $s_{ij} < 0.70$ erhalten Gewicht 0


In [ ]:
# Gemeinsame Dimensionen zwischen onet_pivot und mu_df
mu_active  = mu_df[mu_df["exposure_type"] != "negligible"].set_index("element_name")["mu_i"]
common_dims = [c for c in onet_pivot.columns if c in mu_active.index]
print(f"Gemeinsame Fähigkeitsdimensionen für Indexberechnung: {len(common_dims)}")

onet_matrix = onet_pivot.set_index("soc_code")[common_dims].fillna(0)
mu_vector   = mu_active[common_dims].values

# LLM-Capability-Referenzprofil (normierter μ-Vektor)
llm_profile = mu_vector / (np.linalg.norm(mu_vector) + 1e-9)

# ── Skill-Overlap-Koeffizient s_ij (Pearson-Korrelation) ──────────────────
def skill_overlap(row_vals, reference):
    if np.std(row_vals) < 1e-9:
        return 0.0
    r, _ = pearsonr(row_vals, reference)
    return max(r, 0.0)

sij_values = onet_matrix.apply(lambda row: skill_overlap(row.values, llm_profile), axis=1)
sij_mask   = (sij_values >= SKILL_OVERLAP_THRESHOLD).astype(float)
print(f"Berufe mit s_ij ≥ {SKILL_OVERLAP_THRESHOLD}: {int(sij_mask.sum())} von {len(sij_mask)}")

# ── E_j berechnen ──────────────────────────────────────────────────────────
weighted_sum = onet_matrix.dot(mu_vector)
E_j_raw      = weighted_sum * sij_mask

# Subindizes (NaN-sicher via np.where)
mu_vals = mu_active[common_dims].values
mu_sub  = np.where(mu_vals > 0.7,                              mu_vals, 0)
mu_aug  = np.where((mu_vals >= 0.3) & (mu_vals <= 0.7),        mu_vals, 0)

E_sub_raw = onet_matrix.dot(mu_sub) * sij_mask
E_aug_raw = onet_matrix.dot(mu_aug) * sij_mask

# Min-Max-Normierung auf [0, 1]
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

index_df = pd.DataFrame({
    "soc_code": onet_matrix.index,
    "s_ij":     sij_values.values,
    "E_j":      minmax(E_j_raw).values,
    "E_sub_j":  minmax(E_sub_raw).values,
    "E_aug_j":  minmax(E_aug_raw).values,
})

print(f"\nIndex berechnet: {len(index_df)} Berufe")
print(f"E_j   – Mean: {index_df['E_j'].mean():.3f} | Std: {index_df['E_j'].std():.3f}")
print(f"E^sub – Mean: {index_df['E_sub_j'].mean():.3f} | Std: {index_df['E_sub_j'].std():.3f}")
print(f"E^aug – Mean: {index_df['E_aug_j'].mean():.3f} | Std: {index_df['E_aug_j'].std():.3f}")


Gemeinsame Fähigkeitsdimensionen für Indexberechnung: 49
Berufe mit s_ij ≥ 0.7: 0 von 894

Index berechnet: 894 Berufe
E_j   – Mean: 0.000 | Std: 0.000
E^sub – Mean: 0.000 | Std: 0.000
E^aug – Mean: 0.000 | Std: 0.000


### Monte-Carlo-Konfidenzintervall (Mapping-Konfidenz)

In [24]:
np.random.seed(42)
N_ITER = 1000
mc_results = []

for _ in range(N_ITER):
    mu_noisy = mu_vector * (1 + np.random.normal(0, 0.05, size=len(mu_vector)))
    mu_noisy = np.clip(mu_noisy, 0.1, 0.9)
    E_mc     = minmax(onet_matrix.dot(mu_noisy) * sij_mask)
    mc_results.append(E_mc.values)

mc_array = np.array(mc_results)
ci_half  = np.percentile(mc_array, 97.5, axis=0) - np.percentile(mc_array, 2.5, axis=0)
mean_ci  = ci_half.mean() / 2 * 100

print(f"Monte-Carlo ({N_ITER} Iterationen):")
print(f"  Mittleres 95%-KI: ±{mean_ci:.1f}%  (Zielwert: ±8%)")
print(f"  Status: {'✓ Zielwert erreicht' if mean_ci <= 8 else '⚠ Zielwert überschritten – vgl. Kap. 6.5'}")


Monte-Carlo (1000 Iterationen):
  Mittleres 95%-KI: ±0.0%  (Zielwert: ±8%)
  Status: ✓ Zielwert erreicht


---
## 5.7 Vorbereitung der Analysedatensätze

Datensätze für die drei Analysestufen aus Kapitel 4.8.


In [25]:
# ── Finales Sample zusammenführen ─────────────────────────────────────────
final = (
    index_df
    .merge(train_mapping[["soc_code", "isco_4digit", "mapping_stage"]], on="soc_code", how="inner")
    .merge(ch_isco[["ch_isco_4digit", "ch_isco_label", "main_group"]],
           left_on="isco_4digit", right_on="ch_isco_4digit", how="left")
    .merge(bfs[["ch_isco_4digit", f"employed_{BFS_BASE_YEAR}", f"employed_{BFS_TARGET_YEAR}",
                "delta_bfs", "is_outlier"]],
           on="ch_isco_4digit", how="left")
)

print(f"Finales Analysesample: {len(final)} Berufsgruppen")
print(f"  Hauptgruppe 1 (Führungskräfte):  {(final['main_group']==1).sum()}")
print(f"  Hauptgruppe 2 (Akademisch):      {(final['main_group']==2).sum()}")
print(f"  Hauptgruppe 3 (Techniker):       {(final['main_group']==3).sum()}")
print(f"  Ohne BFS-Match (NaN delta_bfs):  {final['delta_bfs'].isna().sum()}")


Finales Analysesample: 571 Berufsgruppen
  Hauptgruppe 1 (Führungskräfte):  62
  Hauptgruppe 2 (Akademisch):      296
  Hauptgruppe 3 (Techniker):       213
  Ohne BFS-Match (NaN delta_bfs):  21


In [26]:
# ── H1: Gruppenunterschiede nach CH-ISCO-Hauptgruppe ─────────────────────
h1 = final[["soc_code", "isco_4digit", "main_group", "E_j", "E_sub_j", "E_aug_j"]].copy()
h1.to_csv(OUTPUT / "dataset_H1.csv", index=False)
print(f"H1-Datensatz: {len(h1)} Berufe → data/output/dataset_H1.csv ✓")

# ── H2: Fähigkeitsdimensionen als Prädiktoren für E^sub_j ────────────────
h2 = final[["soc_code", "E_sub_j"]].merge(onet_pivot, on="soc_code", how="left")
h2.to_csv(OUTPUT / "dataset_H2.csv", index=False)
print(f"H2-Datensatz: {len(h2)} Berufe × {h2.shape[1]-2} Prädiktoren → data/output/dataset_H2.csv ✓")

# ── H3: Beschäftigungsveränderung ~ E_j + Kontrollvariablen ──────────────
h3 = final[["soc_code", "isco_4digit", "main_group", "E_j", "delta_bfs", "is_outlier",
             f"employed_{BFS_BASE_YEAR}", f"employed_{BFS_TARGET_YEAR}"]].copy()
# Qualifikationsniveau (Proxy via Hauptgruppe; ersetzen durch ISCED wenn verfügbar)
h3["qualification_level"] = h3["main_group"].map({1: 3, 2: 4, 3: 3})
# Sektor-Zuweisung: Platzhalter – manuell oder via NOGA-Mapping ergänzen
h3["sector"] = "Sonstiges"
h3.to_csv(OUTPUT / "dataset_H3.csv", index=False)
print(f"H3-Datensatz: {len(h3)} Berufe → data/output/dataset_H3.csv ✓")
print("  HINWEIS: Sektor-Variable ist Platzhalter – vor Analyse ergänzen!")

# ── Konvergente Validierung (Kuprecht-2025-Vergleich) ─────────────────────
validation = final[["soc_code", "isco_4digit", "E_j", "E_sub_j", "E_aug_j"]].copy()
validation["kuprecht_2025_score"] = np.nan  # Eintragen wenn verfügbar
validation.to_csv(OUTPUT / "dataset_validation.csv", index=False)
print(f"Validierungs-Datensatz → data/output/dataset_validation.csv ✓")

# ── Standardisierte Fähigkeitsvektoren (für Clusteranalyse) ──────────────
skill_std = onet_pivot.set_index("soc_code")[common_dims].copy()
skill_std = (skill_std - skill_std.mean()) / (skill_std.std() + 1e-9)
skill_std.to_csv(ANALYSIS / "skill_vectors_standardized.csv")
print(f"Standardisierte Fähigkeitsvektoren → data/processed/analysis_prep/skill_vectors_standardized.csv ✓")


H1-Datensatz: 571 Berufe → data/output/dataset_H1.csv ✓
H2-Datensatz: 571 Berufe × 161 Prädiktoren → data/output/dataset_H2.csv ✓
H3-Datensatz: 571 Berufe → data/output/dataset_H3.csv ✓
  HINWEIS: Sektor-Variable ist Platzhalter – vor Analyse ergänzen!
Validierungs-Datensatz → data/output/dataset_validation.csv ✓
Standardisierte Fähigkeitsvektoren → data/processed/analysis_prep/skill_vectors_standardized.csv ✓


---
## Finales Sample speichern

`final_sample.csv` ist der zentrale Masterdatensatz für alle Analyse-Notebooks (06_1–06_3).  
Er wird aus dem `final`-DataFrame erstellt, der weiter oben im Notebook durch den  
Merge von Index-Scores, Mapping, CH-ISCO und BFS entstanden ist.


In [27]:
# ── Finales Sample als CSV speichern ──────────────────────────────────────
# Dieser Schritt fehlte bisher — wird von 06_1_eda.ipynb benötigt

final.to_csv(ANALYSIS / "final_sample.csv", index=False)
print(f"Finales Sample gespeichert: {len(final)} Berufe")
print(f"  Spalten: {final.columns.tolist()}")
print(f"  Hauptgruppe 1: {(final['main_group']==1).sum()}")
print(f"  Hauptgruppe 2: {(final['main_group']==2).sum()}")
print(f"  Hauptgruppe 3: {(final['main_group']==3).sum()}")
print(f"  Mit ΔBFS_j:    {final['delta_bfs'].notna().sum()}")
print(f"  Ausreisser:    {final['is_outlier'].sum() if 'is_outlier' in final.columns else 'Spalte fehlt'}")
print()
print("Gespeichert → data/processed/analysis_prep/final_sample.csv ✓")


Finales Sample gespeichert: 571 Berufe
  Spalten: ['soc_code', 's_ij', 'E_j', 'E_sub_j', 'E_aug_j', 'isco_4digit', 'mapping_stage', 'ch_isco_4digit', 'ch_isco_label', 'main_group', 'employed_2022', 'employed_2024', 'delta_bfs', 'is_outlier']
  Hauptgruppe 1: 62
  Hauptgruppe 2: 296
  Hauptgruppe 3: 213
  Mit ΔBFS_j:    550
  Ausreisser:    23

Gespeichert → data/processed/analysis_prep/final_sample.csv ✓


---
## Vollständigkeitsprüfung – alle Outputs vorhanden?

Prüft ob alle Dateien existieren, die von den Analyse-Notebooks benötigt werden.


In [28]:
# ── Vollständigkeitsprüfung ───────────────────────────────────────────────
expected_files = {
    "Notebook 06_1 (EDA)": [
        ANALYSIS / "final_sample.csv",
        ANALYSIS / "skill_vectors_standardized.csv",
    ],
    "Notebook 06_2 (Hypothesen)": [
        ANALYSIS / "final_sample.csv",
        OUTPUT   / "dataset_H1.csv",
        OUTPUT   / "dataset_H2.csv",
        OUTPUT   / "dataset_H3.csv",
        PROCESSED / "onet_pivot.csv",
    ],
    "Notebook 06_3 (Validierung)": [
        ANALYSIS / "final_sample.csv",
        OUTPUT   / "dataset_validation.csv",
    ],
}

all_ok = True
for notebook, files in expected_files.items():
    print(f"\n{notebook}:")
    for f in files:
        exists = f.exists()
        status = "✓" if exists else "✗ FEHLT"
        size   = f"{f.stat().st_size / 1024:.1f} KB" if exists else ""
        print(f"  {status}  {f.name:<45} {size}")
        if not exists:
            all_ok = False

print()
if all_ok:
    print("=" * 55)
    print("  Alle Dateien vorhanden – bereit für Analyse-Notebooks")
    print("=" * 55)
else:
    print("⚠ Fehlende Dateien gefunden – Notebook 1 oder 2 erneut ausführen")



Notebook 06_1 (EDA):
  ✓  final_sample.csv                              86.1 KB
  ✓  skill_vectors_standardized.csv                849.5 KB

Notebook 06_2 (Hypothesen):
  ✓  final_sample.csv                              86.1 KB
  ✓  dataset_H1.csv                                16.8 KB
  ✓  dataset_H2.csv                                1595.8 KB
  ✓  dataset_H3.csv                                51.4 KB
  ✓  onet_pivot.csv                                2566.9 KB

Notebook 06_3 (Validierung):
  ✓  final_sample.csv                              86.1 KB
  ✓  dataset_validation.csv                        16.2 KB

  Alle Dateien vorhanden – bereit für Analyse-Notebooks
